<a href="https://colab.research.google.com/github/Young-Kim-7/Young-Kim-7/blob/main/RepVit_Classification_on_ImageNet-1K_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1: Mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2: GPU 확인
!nvidia-smi

Mon Jun  1 10:47:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 3: RepViT 코드 클론 & 패키지 설치
!git clone https://github.com/THU-MIG/RepVit.git
%cd RepVit
!pip install -r requirements.txt

Cloning into 'RepVit'...
remote: Enumerating objects: 374, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 374 (delta 81), reused 64 (delta 64), pack-reused 272 (from 1)
Receiving objects: 100% (374/374), 20.74 MiB | 19.94 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/content/RepVit
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 20.8 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=77c4e131efdc37d903dee1d5a2957b6e85807135f61be5e672424b8c69ce9131
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for iopath: filename=iopath-0.

In [ ]:
# 4. 압축 해제
!mkdir -p /content/imagenet/val
! tar -xf /content/drive/MyDrive/ILSVRC2012_img_val.tar -C /content/imagenet/val

In [ ]:
# 5. 클래스별 폴더 정리
!wget https://raw.githubusercontent.com/soumith/imagenetloader.torch/master/valprep.sh
!mv valprep.sh /content/imagenet/val/
!cd /content/imagenet/val && bash valprep.sh

--2026-06-01 10:51:50--  https://raw.githubusercontent.com/soumith/imagenetloader.torch/master/valprep.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2220000 (2.1M) [text/plain]
Saving to: ‘valprep.sh’

valprep.sh          100%[===================>]   2.12M  --.-KB/s    in 0.03s   

2026-06-01 10:51:50 (66.1 MB/s) - ‘valprep.sh’ saved [2220000/2220000]



In [ ]:
# 6. 폴더 구조 확인
import os
val_dir = '/content/imagenet/val'
folders = [f for f in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, f))]
print(f"클래스 폴더 수: {len(folders)}")  # 1000 이 나와야 정상

클래스 폴더 수: 1000


In [ ]:
# 7. RepVit-M Series Pretrained 모델별 다운로드 (only 300 epochs)
!mkdir pretrain

# M0.9
!wget -P pretrain https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m0_9_distill_300e.pth

# M1.0
!wget -P pretrain https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m1_0_distill_300e.pth

# M1.1
!wget -P pretrain https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m1_1_distill_300e.pth

# M1.5
!wget -P pretrain https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m1_5_distill_300e.pth

# M2.3
!wget -P pretrain https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m2_3_distill_300e.pth

print("모든 모델 다운로드 완료")

--2026-06-01 10:58:15--  https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m0_9_distill_300e.pth
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/667632599/0fb45a90-f855-40fe-b501-de1d2a459b0e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-01T11%3A37%3A44Z&rscd=attachment%3B+filename%3Drepvit_m0_9_distill_300e.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-01T10%3A36%3A48Z&ske=2026-06-01T11%3A37%3A44Z&sks=b&skv=2018-11-09&sig=0Zf3fd4O4OdpqspKoYyYJq9Q4TLDgErEMyrWVObRSng%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MDMxMzI5NSwibmJmIjoxNzgwMzExNDk1LCJwYXRoIjoicmVsZWFzZWFzc2V

In [ ]:
from PIL import Image
import os
import subprocess
import re

# dummy 이미지 생성 (추론 전 필수!)
os.makedirs('/content/imagenet/train/dummy', exist_ok=True)
img = Image.new('RGB', (224, 224))
img.save('/content/imagenet/train/dummy/dummy.JPEG')
print("dummy 이미지 생성 완료!")

# 검증할 모델 목록
models = [
    ("repvit_m0_9", "pretrain/repvit_m0_9_distill_300e.pth", 78.7),
    ("repvit_m1_0", "pretrain/repvit_m1_0_distill_300e.pth", 80.0),
    ("repvit_m1_1", "pretrain/repvit_m1_1_distill_300e.pth", 80.7),
    ("repvit_m1_5", "pretrain/repvit_m1_5_distill_300e.pth", 82.3),
    ("repvit_m2_3", "pretrain/repvit_m2_3_distill_300e.pth", 83.3),
]

results = []

for model_name, ckpt_path, paper_acc in models:
    print(f"\n{'='*50}")
    print(f"현재 실행 중: {model_name}")
    print(f"{'='*50}")

    cmd = f"""python main.py \
        --eval \
        --model {model_name} \
        --resume {ckpt_path} \
        --data-path /content/imagenet"""

    output = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    stdout = output.stdout + output.stderr

    # 결과에서 Acc@1 추출
    match = re.search(r'Acc@1\s+([\d.]+)', stdout)
    if match:
        acc1 = float(match.group(1))
        diff = acc1 - paper_acc
        results.append((model_name, paper_acc, acc1, diff))
        print(f"완료! Acc@1: {acc1:.3f}%")
    else:
        print(f"결과 추출 실패")
        print(stdout[-500:])

# 최종 결과 출력
print(f"\n{'='*50}")
print(f"{'모델':<15} {'논문':>8} {'재현':>8} {'차이':>8}")
print(f"{'='*50}")
for model_name, paper_acc, acc1, diff in results:
    print(f"{model_name:<15} {paper_acc:>7.1f}% {acc1:>7.3f}% {diff:>+7.3f}%")
print(f"{'='*50}")

dummy 이미지 생성 완료!

현재 실행 중: repvit_m0_9
완료! Acc@1: 78.728%

현재 실행 중: repvit_m1_0
완료! Acc@1: 80.014%

현재 실행 중: repvit_m1_1
완료! Acc@1: 80.678%

현재 실행 중: repvit_m1_5
완료! Acc@1: 82.284%

현재 실행 중: repvit_m2_3
완료! Acc@1: 83.316%

모델                    논문       재현       차이
repvit_m0_9        78.7%  78.728%  +0.028%
repvit_m1_0        80.0%  80.014%  +0.014%
repvit_m1_1        80.7%  80.678%  -0.022%
repvit_m1_5        82.3%  82.284%  -0.016%
repvit_m2_3        83.3%  83.316%  +0.016%
